# Metriche di complessità anno per anno

Per ogni anno e risoluzione calcola, sulle Mcp allineate e binarizzate (HS02):

| algoritmo | country | product |
|---|---|---|
| `extended_fitness` (extended fitness con import) | F, S (import penalty) | Q, CQ, Delta |
| `extended_eci` (extended ECI con import) | F, S | Q, CQ, Delta |
| `fitness` (Tacchella et al.) | F | Q |
| `eci` (Hidalgo–Hausmann) | ECI | PCI |

Le metriche estese usano a = ρα = `A` (default a = 1), C classica HS02 (la stessa per tutti gli anni), `tensor_window = TENSOR_WINDOW`.

Output (un file per risoluzione × algoritmo × tipo × anno, vedi `src/storage.py`):
- `results/econometria/metriche/{res}_digits/{algo}/country/Y{year}.parquet`: `year, country, iso3, <variabili>`
- `results/econometria/metriche/{res}_digits/{algo}/product/Y{year}.parquet`: `year, product, <variabili>`

Gli anni già calcolati (tutti e 8 i file presenti) vengono saltati, a meno di `OVERWRITE = True`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path().resolve(), *Path().resolve().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

from src.algorithms import compute_alpha, compute_fitness, compute_eci
from src.algorithms_with_imports import compute_extended_fitness_with_imports, compute_extended_eci_with_imports
from src.storage import ALGOS, KINDS, YEARS, metric_path, save_metric

## Configurazione

In [2]:
RESOLUTIONS = [4, 6]
A = 1.0
TENSOR_WINDOW = 3
OVERWRITE = False

DATA = ROOT / "data" / "HS02_all_years"
iso3 = pd.read_csv(ROOT / "data" / "raw" / "BACI" / "country_codes_V202601.csv").set_index("country_code")["country_iso3"]

## Calcolo

In [3]:
EXTENDED = {"extended_fitness": compute_extended_fitness_with_imports,
            "extended_eci":     compute_extended_eci_with_imports}

def metriche_anno(res, year, C, alpha):
    """{(algo, kind): tabella larga} per un anno."""
    base = DATA / f"{res}_digits" / "alligned" / "binarized"
    Me = pd.read_parquet(base / "Mcp_export" / f"Mcp_export_alligned_Y{year}.parquet")
    Mi = pd.read_parquet(base / "Mcp_import" / f"Mcp_import_alligned_Y{year}.parquet")
    out = {}

    for name, algo in EXTENDED.items():
        Q, F, CQ, Delta, S = algo(Me, Mi, C.T, alpha=alpha, tensor_window=TENSOR_WINDOW, max_iter=5000,
                                  tol=1e-6, track_convergence=False, save_delta=False, delta_window=None)
        out[(name, "country")] = {"F": F, "S": S}
        out[(name, "product")] = {"Q": Q, "CQ": CQ, "Delta": Delta}

    F, Q = compute_fitness(Me, track_convergence=False)
    out[("fitness", "country")], out[("fitness", "product")] = {"F": F}, {"Q": Q}
    eci, pci = compute_eci(Me)
    out[("eci", "country")], out[("eci", "product")] = {"ECI": eci}, {"PCI": pci}

    tables = {}
    for (algo, kind), cols in out.items():
        t = pd.DataFrame(cols).rename_axis(kind).reset_index().assign(year=year)
        if kind == "country":
            t["iso3"] = t["country"].map(iso3)
        tables[(algo, kind)] = t
    return tables

for res in RESOLUTIONS:
    C = pd.read_parquet(DATA / f"{res}_digits" / "alligned" / "binarized" / "C" / f"C_{res}_digits_HS02.parquet")
    alpha = A * compute_alpha(C)               # compute_alpha = 1/rho  ->  alpha = a / rho
    for year in YEARS:
        if not OVERWRITE and all(metric_path(res, a, k, year).exists() for a in ALGOS for k in KINDS):
            continue
        tables = metriche_anno(res, year, C, alpha)
        for (algo, kind), t in tables.items():
            save_metric(t, res, algo, kind, year)
        print(f"{res} digits, {year}: {len(tables[('eci', 'country')])} paesi, "
              f"{len(tables[('eci', 'product')])} prodotti", flush=True)

C:\Users\vitto\Desktop\CSH RESEARCH\src\algorithms.py:151: RuntimeWarning: divide by zero encountered in divide
  Q = 1 / (M.T @ (1 / F))


6 digits, 2003: 222 paesi, 5079 prodotti


C:\Users\vitto\Desktop\CSH RESEARCH\src\algorithms.py:151: RuntimeWarning: divide by zero encountered in divide
  Q = 1 / (M.T @ (1 / F))
C:\Users\vitto\Desktop\CSH RESEARCH\src\algorithms.py:151: RuntimeWarning: overflow encountered in divide
  Q = 1 / (M.T @ (1 / F))
C:\Users\vitto\Desktop\CSH RESEARCH\src\algorithms.py:151: RuntimeWarning: invalid value encountered in matmul
  Q = 1 / (M.T @ (1 / F))


6 digits, 2004: 222 paesi, 5079 prodotti


C:\Users\vitto\Desktop\CSH RESEARCH\src\algorithms.py:151: RuntimeWarning: divide by zero encountered in divide
  Q = 1 / (M.T @ (1 / F))
C:\Users\vitto\Desktop\CSH RESEARCH\src\algorithms.py:151: RuntimeWarning: overflow encountered in divide
  Q = 1 / (M.T @ (1 / F))
C:\Users\vitto\Desktop\CSH RESEARCH\src\algorithms.py:151: RuntimeWarning: invalid value encountered in matmul
  Q = 1 / (M.T @ (1 / F))


...

6 digits, 2022: 226 paesi, 5079 prodotti


C:\Users\vitto\Desktop\CSH RESEARCH\src\algorithms.py:151: RuntimeWarning: divide by zero encountered in divide
  Q = 1 / (M.T @ (1 / F))


6 digits, 2023: 226 paesi, 5079 prodotti


C:\Users\vitto\Desktop\CSH RESEARCH\src\algorithms.py:151: RuntimeWarning: divide by zero encountered in divide
  Q = 1 / (M.T @ (1 / F))


6 digits, 2024: 226 paesi, 5079 prodotti
